# 🏛️ JullBrain V3.0 — Legal-RAG & Informática Forense
### Pipeline de Extracción Jurídica con IBM Docling y Embeddings IBM Granite R2 en Google Colab (GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julljoll/JullBrain/blob/main/JullBrain_Colab.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-JullBrain-blue.svg)](https://github.com/julljoll/JullBrain.git)
[![IBM Granite](https://img.shields.io/badge/IBM-Granite%203.3%208B-purple.svg)](https://huggingface.co/ibm-granite)
[![License](https://img.shields.io/badge/License-Apache%202.0-yellow.svg)](https://opensource.org/licenses/Apache-2.0)

---

In [ ]:
# @title 0. 🎨 Banner Informativo de JullBrain V3.0
from IPython.display import display, HTML

banner_html = """
<div style="background: linear-gradient(135deg, #0d1117 0%, #161b22 100%); border: 1px solid #30363d; border-radius: 10px; padding: 20px; font-family: 'Segoe UI', system-ui, sans-serif; color: #f0f6fc; box-shadow: 0 4px 15px rgba(0,0,0,0.5);">
    <div style="display: flex; align-items: center; justify-content: space-between; border-bottom: 1px solid #21262d; padding-bottom: 12px; margin-bottom: 15px;">
        <span style="font-size: 24px; font-weight: bold; color: #f2cc60;">⚖️ JullBrain Legal-RAG V3.0</span>
        <span style="background: #238636; color: #ffffff; padding: 4px 10px; border-radius: 20px; font-size: 12px; font-weight: bold;">GPU MODE</span>
    </div>
    <p style="color: #8b949e; font-size: 14px; margin: 0 0 12px 0;">
        Sistema Enterprise de Procesamiento Documental, Estructuración Jerárquica y Vectorización de Leyes para Inteligencias Artificiales de <strong>IBM Granite</strong>.
    </p>
    <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 10px; font-size: 13px;">
        <div style="background: #0d1117; padding: 10px; border-radius: 6px; border: 1px solid #30363d;">
            <strong style="color: #58a6ff;">📄 Motor OCR:</strong> IBM Docling Hierarchical
        </div>
        <div style="background: #0d1117; padding: 10px; border-radius: 6px; border: 1px solid #30363d;">
            <strong style="color: #58a6ff;">🧠 Embeddings:</strong> Granite Embedding R2
        </div>
        <div style="background: #0d1117; padding: 10px; border-radius: 6px; border: 1px solid #30363d;">
            <strong style="color: #58a6ff;">🔐 Integridad:</strong> SHA-256 & Error Cero
        </div>
    </div>
</div>
"""
display(HTML(banner_html))

## 1. ⚙️ Diagnóstico de Aceleración por GPU
Verifica que el entorno de ejecución tenga una GPU asignada (por ejemplo, NVIDIA T4 o A100).

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU activa: {torch.cuda.get_device_name(0)}")
    print(f"✅ Memoria VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")
else:
    print("⚠️ ADVERTENCIA: No se detectó GPU. Ve a 'Entorno de ejecución' -> 'Cambiar tipo de entorno' y selecciona GPU T4.")

## 2. 🔄 Sincronización Automática con GitHub
Clona el repositorio si no existe, o descarga automáticamente las últimas actualizaciones con `git pull`.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/julljoll/JullBrain.git"
REPO_DIR = Path("/content/JullBrain")

if not REPO_DIR.exists():
    print(f"📥 Clonando repositorio desde {REPO_URL}...")
    !git clone {REPO_URL} /content/JullBrain
    %cd /content/JullBrain
else:
    %cd /content/JullBrain
    print("🔄 Sincronizando con los últimos cambios de GitHub...")
    !git pull origin main

print(f"\n📁 Directorio de trabajo actual: {os.getcwd()}")

## 3. 📦 Instalación de Dependencias Optimizadas para Colab
Instala silenciosamente las dependencias de producción definidas en `requirements_colab.txt`.

In [ ]:
!pip install -q -r requirements_colab.txt

print("✅ Dependencias instaladas y verificadas:")
import docling
import fitz  # PyMuPDF
import sentence_transformers
print(f" • IBM Docling: {docling.__version__}")
print(f" • PyMuPDF: {fitz.__version__}")
print(f" • Sentence-Transformers: {sentence_transformers.__version__}")

## 4. ☁️ Montaje de Google Drive (Persistencia Permanente)
Permite conservar las leyes procesadas y cargar PDFs adicionales directamente desde tu Drive.

In [ ]:
from colab_runner import JullBrainColabRunner

runner = JullBrainColabRunner()
runner.imprimir_resumen_entorno()
runner.montar_google_drive()
runner.sincronizar_pdfs_desde_drive()

## 5. 🔍 Inspección del Corpus Jurídico en `RAG/datos_brutos`
Enumera los documentos PDF organizados por rama del Derecho (Penal, Civil, Administrativo, Constitucional, etc.).

In [ ]:
from pathlib import Path
from rich.table import Table
from rich.console import Console

console = Console()
table = Table(title="🏛️ Corpus Normativo Local JullBrain", show_header=True, header_style="bold magenta")
table.add_column("Materia / Categoría", style="cyan", width=25)
table.add_column("Documentos PDF", justify="right", style="green")
table.add_column("Estado", style="yellow")

base_brutos = Path("RAG/datos_brutos")
total_pdfs = 0

if base_brutos.exists():
    for cat in sorted(base_brutos.iterdir()):
        if cat.is_dir():
            pdfs = list(cat.glob("*.pdf"))
            total_pdfs += len(pdfs)
            table.add_row(cat.name, str(len(pdfs)), "✅ Listo" if pdfs else "Vacío")

console.print(table)
print(f"\nTotal de normativas detectadas en bruto: {total_pdfs} archivos PDF.")

## 6. 🏛️ Ejecución del Pipeline: Conversión con IBM Docling
Realiza la extracción inteligente, análisis de layout, OCR y chunking jerárquico acelerado por GPU.

In [ ]:
# Ejecución del pipeline maestro de conversión
!python convert_normativas.py

## 7. ⚖️ Control de Jerarquía Legal Venezolana (Legalizer)
Normaliza la estructura: `LIBRO` > `TÍTULO` > `CAPÍTULO` > `SECCIÓN` > `Artículo` > `Parágrafo`.

In [ ]:
# Aplica el post-procesado legal sobre el Markdown generado
!python convert_normativas.py --post-process-only

## 8. 🔬 Doble Validación Cruzada (Principio Forense de Error Cero)
Compara los artículos detectados por regex con la extracción de bajo nivel de PyMuPDF.

In [ ]:
# Validador cruzado PDF original vs Markdown generado
!python validator.py

## 9. 🧠 Vectorización con Embeddings de IBM Granite R2 (768 dimensiones)
Genera embeddings semánticos para cada artículo normativo utilizando la GPU NVIDIA T4.

In [ ]:
# Genera y verifica embeddings semánticos con IBM Granite
!python granite_embedder.py

## 10. 🔐 Generación del Manifiesto Criptográfico Forense (SHA-256)
Certifica la cadena de custodia y garantiza que ningún artículo haya sido alterado.

In [ ]:
manifiesto = runner.generar_manifiesto_sha256()
print(f"Total de normativas selladas criptográficamente: {len(manifiesto)}")

## 11. ☁️ Respaldo Automático a Google Drive
Sincroniza todos los Markdown estructurados, JSONs de chunks y vectores a Google Drive.

In [ ]:
runner.respaldar_en_google_drive()

## 12. 🧪 Suite de QA Testing y Verificación de Integridad
Ejecuta las pruebas unitarias y de integración offline del sistema.

In [ ]:
!python -m unittest discover tests

## 13. 💬 Consulta Interactiva al Sistema Legal-RAG con IBM Granite
Prueba una consulta jurídica real anclada al corpus normativo cerrado.

In [ ]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

# Consulta jurídica de demostración
consulta = "¿Cuáles son las formalidades requeridas para la recolección y aseguramiento de la evidencia digital según la ley venezolana?"

print(f"🔎 Consultando: {consulta}\n")
response = client.post("/api/v1/query", json={
    "notebook_id": "default_notebook",
    "user_message": consulta,
    "top_k": 5
})

if response.status_code == 200:
    data = response.json()
    print("=" * 80)
    print("⚖️ DICTAMEN LEGAL-RAG (IBM GRANITE):")
    print("=" * 80)
    print(data.get("response", "Sin respuesta generada."))
    print("=" * 80)
else:
    print(f"Respuesta API: {response.status_code} - {response.text}")

## 14. 📤 Exportación para Bases de Datos Vectoriales y RAGFlow
Genera un archivo `.jsonl` listo para ser importado por el motor de base de datos vectorial de RAGFlow.

In [ ]:
import json
from pathlib import Path

chunks_dir = Path("normativas_chunks")
export_file = Path("normativas_ragflow_export.jsonl")
contador = 0

with open(export_file, "w", encoding="utf-8") as out:
    for cf in chunks_dir.rglob("*.json"):
        try:
            with open(cf, "r", encoding="utf-8") as f:
                doc = json.load(f)
                if isinstance(doc, list):
                    for item in doc:
                        out.write(json.dumps(item, ensure_ascii=False) + "\n")
                        contador += 1
                elif isinstance(doc, dict):
                    out.write(json.dumps(doc, ensure_ascii=False) + "\n")
                    contador += 1
        except Exception:
            pass

print(f"✅ Exportación completada: {contador} chunks normativos guardados en {export_file}")